In [0]:
from pyspark.sql import functions as F
import pyspark.sql.types as T


In [0]:
s3_path = "s3://databricks-free-portfolio-586794464819-us-east-2-an/manual_loads"
bad_records = f"{s3_path}/sales_BadRecords"
spark.read.csv(f"{s3_path}/mock_sales_data.csv", header=True, inferSchema=True).show(5)


In [0]:
sales_schema = (
    T.StructType()
    .add("transaction_id", T.IntegerType())
    .add("customer_name", T.StringType())
    .add("purchase_date", T.DateType())
    .add("product_category", T.StringType())
    .add("amount_usd", T.DoubleType())
    .add("store_location", T.StringType())
)

store_schema = (
    T.StructType()
    .add("store_id", T.StringType())
    .add("store_name", T.StringType())
    .add("region", T.StringType())
    .add("manager_name", T.StringType())
    .add("opened_year", T.IntegerType())
    .add("is_active", T.BooleanType())
)

sales_df = spark.read.schema(sales_schema).options(dateFormat = "M/d/yyyy", badRecordsPath=f"{bad_records}/fact_sales").csv(f"{s3_path}/mock_sales_data.csv", header=True)
                                                                                                            
store_df = spark.read.schema(store_schema).option("badRecordsPath",f"{bad_records}/dim_store").csv(f"{s3_path}/dim_store.csv", header=True, inferSchema=False)


sales_df.show(5)
store_df.show(5)

#Show bad records
print("-------------------BAD RECORDS---------------------")
spark.read.csv(
    f"{bad_records}/*/*/*"
).show(5, truncate=False)

In [0]:
store_df.filter(F.col("region") == "North America").limit(5).show()

sales_df = sales_df.withColumn("store_id",(
    F.when(F.col("store_location") == "New York", "STR_003")
    .when(F.col("store_location") == "Georgia", "STR_007")
    .otherwise(""))
)

#check to see if there are multiple store ids for a given store location
store_df.groupBy("store_id").agg(F.count("region")).orderBy("store_id")

In [0]:
sales_agg = sales_df.groupBy("store_location", "store_id", "purchase_date").agg(F.sum("amount_usd").alias("total_sales"), F.count("transaction_id").alias("total_transactions"))

sales_w_store = sales_agg.join(store_df.select("store_id", "store_name", "manager_name", "region"), sales_agg.store_id == store_df.store_id, "left")

sales_w_store.show(5)



In [0]:
if spark.catalog.tableExists("`dataexpert-portfolio`.demo_artifacts.`gold_sales_by_region`"):
  pass
else: 
    sales_agg.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("`dataexpert-portfolio`.demo_artifacts.gold_sales_by_region")